<a href="https://colab.research.google.com/github/danadorn/404-/blob/main/lab2-multiple-linear-regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Lab 2: Multiple Linear Regression - Car Price Prediction

**Student Name:** _________________________

---

##  Scenario

You are a **Machine Learning Engineer** at **Khmer99.com**, a leading car marketplace website. The company wants you to develop a model that can **predict the price of used cars** based on their features.

Your model will help:
- Sellers price their cars competitively
- Buyers identify fair deals
- The company provide better service to customers

---

##  Learning Objectives

By completing this lab, you will:
1. Build a **multiple linear regression** model (many features → one target)
2. Handle **categorical variables** (encoding)
3. Perform **feature engineering** and selection
4. Evaluate model performance comprehensively
5. Interpret **feature importance**
6. Compare **different approaches**

---

##  Dataset Description

The dataset contains information about 100 used cars with the following features:

| Feature | Description | Type |
|---------|-------------|------|
| **Car_ID** | Unique identifier | Identifier |
| **Brand** | Manufacturer (Toyota, Honda, etc.) | Categorical |
| **Model** | Model name (Corolla, Civic, etc.) | Categorical |
| **Year** | Manufacturing year | Numerical |
| **Kilometers_Driven** | Total distance driven | Numerical |
| **Fuel_Type** | Petrol, Diesel, or Electric | Categorical |
| **Transmission** | Manual or Automatic | Categorical |
| **Owner_Type** | First, Second, or Third owner | Categorical |
| **Mileage** | Fuel efficiency (km/liter) | Numerical |
| **Engine** | Engine capacity (CC) | Numerical |
| **Power** | Maximum power (bhp) | Numerical |
| **Seats** | Number of seats | Numerical |
| **Price**  | Selling price in USD (**TARGET**) | Numerical |

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##  Step 0: Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print(" All libraries imported successfully!")

 All libraries imported successfully!


---
##  Step 1: Load and Explore Data

### 1.1 Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('cars.csv')

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")

# Display first few rows
df.head()

### 1.2 Basic Information

In [ ]:
# Data types and missing values
print("Dataset Information:")
print("="*60)
df.info()

print("\n" + "="*60)
print("Missing Values:")
print("="*60)
print(df.isnull().sum())

print("\n" + "="*60)
print("Unique Values per Column:")
print("="*60)
for col in df.columns:
    print(f"{col:20s}: {df[col].nunique():3d} unique values")

### 1.3 Statistical Summary

In [ ]:
# Statistical summary of numerical features
print("Numerical Features Summary:")
print("="*80)
df.describe().round(2)

In [ ]:
# Summary of categorical features
print("\nCategorical Features Distribution:")
print("="*60)

categorical_cols = ['Brand', 'Fuel_Type', 'Transmission', 'Owner_Type']
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print("-" * 40)

** Initial Observations:**
- Dataset has both numerical and categorical features
- Price ranges from \$5,625 to \$50,000
- Year ranges from 2016 to 2021 (relatively new cars)
- Multiple brands represented
- Mix of petrol and diesel cars
- Both manual and automatic transmissions

---
##  Step 2: Exploratory Data Analysis (EDA)

### 2.1 Target Variable Distribution

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Price'], bins=20, color='#028090', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Price ($)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Car Prices', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(df['Price'], vert=True)
axes[1].set_ylabel('Price ($)', fontsize=12, fontweight='bold')
axes[1].set_title('Price Box Plot', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Price Statistics:")
print(f"  Mean: ${df['Price'].mean():,.2f}")
print(f"  Median: ${df['Price'].median():,.2f}")
print(f"  Std Dev: ${df['Price'].std():,.2f}")

### 2.2 Numerical Features Analysis

In [ ]:
# Select numerical features (excluding ID and Price)
numerical_features = ['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats']

# Create subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, feature in enumerate(numerical_features):
    axes[idx].scatter(df[feature], df['Price'], alpha=0.6, color='#028090', edgecolors='black', s=60)
    axes[idx].set_xlabel(feature, fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Price ($)', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'Price vs. {feature}', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

    # Add correlation coefficient
    corr = df[feature].corr(df['Price'])
    axes[idx].text(0.05, 0.95, f'Correlation: {corr:.3f}',
                  transform=axes[idx].transAxes,
                  fontsize=10, verticalalignment='top',
                  bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

### 2.3 Correlation Matrix

In [ ]:
# Calculate correlation matrix for numerical features
numerical_df = df[numerical_features + ['Price']]
correlation_matrix = numerical_df.corr()

# Visualize
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=2, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Numerical Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Show correlations with Price
print("\nCorrelations with Price (sorted):")
print("="*50)
price_corr = correlation_matrix['Price'].sort_values(ascending=False)
for feature, corr in price_corr.items():
    if feature != 'Price':
        print(f"{feature:20s}: {corr:6.3f}")

### 2.4 Categorical Features Analysis

In [ ]:
# Analyze price by categorical features
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
categorical_cols = ['Fuel_Type', 'Transmission', 'Owner_Type', 'Brand']

for idx, col in enumerate(categorical_cols):
    row = idx // 2
    col_idx = idx % 2

    if col == 'Brand':
        # For Brand, show only top 10 by count
        top_brands = df['Brand'].value_counts().head(10).index
        data_to_plot = df[df['Brand'].isin(top_brands)]
        sns.boxplot(data=data_to_plot, y='Brand', x='Price', ax=axes[row, col_idx], palette='Set2')
    else:
        sns.boxplot(data=df, x=col, y='Price', ax=axes[row, col_idx], palette='Set2')

    axes[row, col_idx].set_title(f'Price by {col}', fontsize=13, fontweight='bold')
    axes[row, col_idx].set_xlabel(col, fontsize=11, fontweight='bold')
    axes[row, col_idx].set_ylabel('Price ($)', fontsize=11, fontweight='bold')
    axes[row, col_idx].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

** EDA Insights:**
1. **Power** and **Engine** have strongest positive correlations with Price
2. **Year** also shows positive correlation (newer cars cost more)
3. **Kilometers_Driven** has slight negative correlation (more km = lower price)
4. **Automatic** transmission cars tend to be pricier than manual
5. Luxury brands (Mercedes, BMW, Audi) have higher prices

---
##  Step 3: Feature Engineering & Preprocessing

### 3.1 Create Derived Features

Let's create some potentially useful features:

In [ ]:
# Create a copy for feature engineering
df_engineered = df.copy()

# 1. Car Age (more intuitive than Year)
current_year = 2024
df_engineered['Car_Age'] = current_year - df_engineered['Year']

# 2. Power to Weight ratio (using Engine as proxy for weight)
df_engineered['Power_per_CC'] = df_engineered['Power'] / df_engineered['Engine']

# 3. Luxury brand indicator
luxury_brands = ['Mercedes', 'BMW', 'Audi']
df_engineered['Is_Luxury'] = df_engineered['Brand'].isin(luxury_brands).astype(int)

print(" New features created:")
print("   1. Car_Age: Age of the car in years")
print("   2. Power_per_CC: Power-to-displacement ratio")
print("   3. Is_Luxury: Binary indicator for luxury brands")
print(f"\nNew dataset shape: {df_engineered.shape}")

### 3.2 Handle Categorical Variables

**Problem:** Linear regression requires numerical inputs, but we have categorical features.

**Solutions:**
1. **Label Encoding**: Convert categories to numbers (0, 1, 2, ...)
2. **One-Hot Encoding**: Create binary columns for each category

We'll use Label Encoding for simplicity (suitable for ordinal categories).

In [ ]:
# Copy for encoding
df_encoded = df_engineered.copy()

# Label encode categorical variables
label_encoders = {}
categorical_columns = ['Brand', 'Model', 'Fuel_Type', 'Transmission', 'Owner_Type']

print("Encoding categorical variables:")
print("="*60)

for col in categorical_columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le
    print(f"{col:15s}: {len(le.classes_)} categories encoded")

print("\n All categorical variables encoded!")
print("\nSample of encoded data:")
df_encoded[categorical_columns].head()

### 3.3 Select Features for Modeling

We'll drop irrelevant columns and separate features from target.

In [ ]:
# Drop Car_ID (not predictive) and Model (too many unique values)
features_to_drop = ['Car_ID', 'Model', 'Year']  # Year replaced by Car_Age
df_model = df_encoded.drop(columns=features_to_drop)

# Separate features (X) and target (y)
X = df_model.drop('Price', axis=1)
y = df_model['Price']

print("Features for Modeling:")
print("="*60)
print(f"Number of features: {X.shape[1]}")
print(f"Features: {list(X.columns)}")
print(f"\nTarget: Price")
print(f"\nFinal dataset shape:")
print(f"  X (features): {X.shape}")
print(f"  y (target):   {y.shape}")

---
##  Step 4: Train-Test Split

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Data Split Summary:")
print("="*60)
print(f"Training set:  {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set:      {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")
print(f"Total:         {len(X)} samples")
print("="*60)
print("\n Data successfully split!")

---
##  Step 5: Build Linear Regression Model

### 5.1 Train the Model

**Multiple Linear Regression Equation:**

```
Price = w₀ + w₁×Feature₁ + w₂×Feature₂ + ... + wₙ×Featureₙ
```

Where each feature has its own coefficient (weight).

In [ ]:
# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

print(" Model trained successfully!")
print("\n" + "="*80)
print("MODEL PARAMETERS:")
print("="*80)
print(f"Intercept (w₀): ${model.intercept_:,.2f}")
print(f"Number of coefficients: {len(model.coef_)}")
print("="*80)

### 5.2 Feature Importance Analysis

Let's see which features have the biggest impact on price!

In [ ]:
# Create feature importance DataFrame
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_,
    'Abs_Coefficient': np.abs(model.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print("\n FEATURE IMPORTANCE (by coefficient magnitude):")
print("="*70)
print(feature_importance.to_string(index=False))
print("="*70)

# Visualize
plt.figure(figsize=(12, 6))
colors = ['#02C39A' if x > 0 else '#DC2626' for x in feature_importance['Coefficient']]
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors, alpha=0.7, edgecolor='black')
plt.xlabel('Coefficient Value', fontsize=12, fontweight='bold')
plt.title('Feature Importance (Coefficient Values)', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=1)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n INTERPRETATION:")
print("  Green bars (positive): Feature increases price")
print("  Red bars (negative): Feature decreases price")
print("  Longer bars: Stronger impact on price")

---
##  Step 6: Model Evaluation

### 6.1 Make Predictions

In [ ]:
# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Create comparison DataFrame for test set
comparison_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_test_pred,
    'Error': y_test.values - y_test_pred,
    'Abs_Error': np.abs(y_test.values - y_test_pred),
    'Percent_Error': np.abs((y_test.values - y_test_pred) / y_test.values * 100)
}).round(2)

print("\n SAMPLE PREDICTIONS (First 10 test examples):")
print("="*80)
print(comparison_df.head(10).to_string(index=False))
print("="*80)

### 6.2 Calculate Metrics

In [ ]:
# Calculate metrics for both sets
def calculate_metrics(y_true, y_pred, set_name=""):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R²': r2,
        'MAPE': mape
    }

train_metrics = calculate_metrics(y_train, y_train_pred, "Train")
test_metrics = calculate_metrics(y_test, y_test_pred, "Test")

# Create metrics comparison table
metrics_df = pd.DataFrame({
    'Metric': ['MAE ($)', 'RMSE ($)', 'R² Score', 'MAPE (%)'],
    'Training Set': [
        f"${train_metrics['MAE']:,.2f}",
        f"${train_metrics['RMSE']:,.2f}",
        f"{train_metrics['R²']:.4f}",
        f"{train_metrics['MAPE']:.2f}%"
    ],
    'Test Set': [
        f"${test_metrics['MAE']:,.2f}",
        f"${test_metrics['RMSE']:,.2f}",
        f"{test_metrics['R²']:.4f}",
        f"{test_metrics['MAPE']:.2f}%"
    ]
})

print("\n" + "="*70)
print(" MODEL PERFORMANCE METRICS")
print("="*70)
print(metrics_df.to_string(index=False))
print("="*70)

### 6.3 Interpret Results

In [ ]:
print("\n METRICS INTERPRETATION:")
print("="*70)

print(f"\n1⃣  Mean Absolute Error (MAE): ${test_metrics['MAE']:,.2f}")
print(f"    → On average, predictions are off by ${test_metrics['MAE']:,.2f}")
print(f"    → This is {test_metrics['MAPE']:.1f}% of the average car price")

print(f"\n2⃣  Root Mean Squared Error (RMSE): ${test_metrics['RMSE']:,.2f}")
print(f"    → Typical prediction error is ${test_metrics['RMSE']:,.2f}")
print(f"    → RMSE > MAE suggests some large errors exist")

print(f"\n3⃣  R² Score: {test_metrics['R²']:.4f}")
print(f"    → Model explains {test_metrics['R²']*100:.2f}% of price variance")
if test_metrics['R²'] > 0.9:
    print("    →  EXCELLENT fit!")
elif test_metrics['R²'] > 0.7:
    print("    →  GOOD fit!")
elif test_metrics['R²'] > 0.5:
    print("    →   MODERATE fit")
else:
    print("    →  POOR fit")

print(f"\n4⃣  Mean Absolute Percentage Error: {test_metrics['MAPE']:.2f}%")
print(f"    → Predictions are {test_metrics['MAPE']:.1f}% off on average")

print(f"\n5⃣  Overfitting Check:")
r2_diff = abs(train_metrics['R²'] - test_metrics['R²'])
if r2_diff < 0.05:
    print(f"    →  No overfitting! (diff: {r2_diff:.4f})")
elif r2_diff < 0.10:
    print(f"    →   Slight overfitting (diff: {r2_diff:.4f})")
else:
    print(f"    →  Significant overfitting! (diff: {r2_diff:.4f})")

print("\n" + "="*70)

### 6.4 Visualizations

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Predicted vs Actual (Training)
axes[0, 0].scatter(y_train, y_train_pred, alpha=0.6, color='#028090', edgecolors='black', s=60)
axes[0, 0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()],
                'r--', lw=3, label='Perfect Prediction')
axes[0, 0].set_xlabel('Actual Price ($)', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Predicted Price ($)', fontsize=12, fontweight='bold')
axes[0, 0].set_title(f'Training Set: Predicted vs Actual (R²={train_metrics["R²"]:.3f})',
                    fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Predicted vs Actual (Test)
axes[0, 1].scatter(y_test, y_test_pred, alpha=0.6, color='#02C39A', edgecolors='black', s=60)
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
                'r--', lw=3, label='Perfect Prediction')
axes[0, 1].set_xlabel('Actual Price ($)', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Predicted Price ($)', fontsize=12, fontweight='bold')
axes[0, 1].set_title(f'Test Set: Predicted vs Actual (R²={test_metrics["R²"]:.3f})',
                    fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Residuals (Test)
residuals = y_test - y_test_pred
axes[1, 0].scatter(y_test_pred, residuals, alpha=0.6, color='#028090', edgecolors='black', s=60)
axes[1, 0].axhline(y=0, color='r', linestyle='--', linewidth=3)
axes[1, 0].set_xlabel('Predicted Price ($)', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Residuals ($)', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Residual Plot (Test Set)', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Residual Distribution
axes[1, 1].hist(residuals, bins=15, color='#02C39A', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=0, color='r', linestyle='--', linewidth=3)
axes[1, 1].set_xlabel('Residuals ($)', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Distribution of Residuals', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
##  Step 7: Make Predictions on New Data

In [ ]:
# Function to predict price for new car
def predict_car_price(features_dict):
    """
    Predict car price given features.

    Parameters:
    -----------
    features_dict : dict
        Dictionary with feature names and values

    Returns:
    --------
    float
        Predicted price
    """
    # Create DataFrame with same columns as training data
    features_df = pd.DataFrame([features_dict], columns=X.columns)
    prediction = model.predict(features_df)[0]
    return prediction

# Example: Predict from first test sample
print("\n EXAMPLE PREDICTION:")
print("="*70)

sample_idx = 0
sample_features = X_test.iloc[sample_idx]
actual_price = y_test.iloc[sample_idx]
predicted_price = y_test_pred[sample_idx]

print("\nInput Features:")
for feature, value in sample_features.items():
    print(f"  {feature:20s}: {value}")

print(f"\nPrediction Results:")
print(f"  Actual Price:     ${actual_price:,.2f}")
print(f"  Predicted Price:  ${predicted_price:,.2f}")
print(f"  Error:            ${abs(actual_price - predicted_price):,.2f}")
print(f"  Accuracy:         {100 - abs((actual_price - predicted_price)/actual_price*100):.1f}%")
print("="*70)

---
##  Step 8: Summary Report

In [ ]:
print("\n" + "="*80)
print(" FINAL MODEL SUMMARY REPORT")
print("="*80)

print("\n MODEL TYPE: Multiple Linear Regression")
print(f"   Number of features: {X.shape[1]}")
print(f"   Target variable: Price")

print("\n DATASET:")
print(f"   Total samples: {len(df)}")
print(f"   Training: {len(X_train)} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"   Testing: {len(X_test)} samples ({len(X_test)/len(df)*100:.1f}%)")

print("\n PERFORMANCE (Test Set):")
print(f"   R² Score:        {test_metrics['R²']:.4f} ({test_metrics['R²']*100:.2f}% variance explained)")
print(f"   RMSE:            ${test_metrics['RMSE']:,.2f}")
print(f"   MAE:             ${test_metrics['MAE']:,.2f}")
print(f"   MAPE:            {test_metrics['MAPE']:.2f}%")

print("\n TOP 3 MOST IMPORTANT FEATURES:")
top3 = feature_importance.head(3)
for idx, row in top3.iterrows():
    impact = "increases" if row['Coefficient'] > 0 else "decreases"
    print(f"   {idx+1}. {row['Feature']:15s} - {impact} price (coef: {row['Coefficient']:,.2f})")

print("\n KEY INSIGHTS:")
print(f"   • Model explains {test_metrics['R²']*100:.1f}% of price variation")
print(f"   • Average prediction error: ${test_metrics['MAE']:,.2f} ({test_metrics['MAPE']:.1f}%)")
print(f"   • Most influential factors: Power, Engine, Brand")
print(f"   • Overfitting: {'Minimal' if r2_diff < 0.05 else 'Moderate' if r2_diff < 0.10 else 'Significant'}")

print("\n CONCLUSION:")
if test_metrics['R²'] > 0.8 and test_metrics['MAPE'] < 15:
    print("   The model performs WELL and can be used for price predictions.")
    print("   Predictions are reliable within the training data range.")
elif test_metrics['R²'] > 0.6:
    print("   The model shows MODERATE performance.")
    print("   Consider adding more features or trying advanced techniques.")
else:
    print("   The model needs improvement.")
    print("   Try: feature engineering, regularization, or different algorithms.")

print("\n" + "="*80)

---
##  Learning Exercises

### Exercise 1: Understanding Multiple Regression (Easy)

**Questions:**
1. How is multiple linear regression different from simple linear regression?
2. Why do we need to encode categorical variables?
3. Name 3 features that increase car price and 3 that decrease it.

### Exercise 2: Model Interpretation (Medium)

**Tasks:**
1. Look at the feature importance plot. Which feature has the strongest positive impact?
2. If a car's Power increases by 50 bhp, how much does the price increase? (Use the coefficient)
3. Compare R² on training vs test set. Is the model overfitting?

### Exercise 3: Experimentation (Hard)

**Try these modifications:**
1. Remove the least important feature and retrain. Does performance improve?
2. Create a new feature (e.g., `Year × Power`) and check if R² improves
3. Change the test_size to 0.3. How do metrics change?

### Exercise 4: Real-World Application (Challenge)

**Scenario:** A customer wants to sell their car:
- Brand: Toyota
- Year: 2020
- Kilometers: 35000
- Fuel: Petrol
- Transmission: Automatic
- Engine: 1500 CC
- Power: 120 bhp
- Seats: 5

**Questions:**
1. How would you encode these features?
2. What price would your model predict?
3. What's the 95% confidence interval? (approx: predicted ± 2×RMSE)

---

##  Key Takeaways

### What You've Learned:

1. **Multiple Linear Regression:**
   - Uses multiple features to predict target
   - More powerful than simple regression
   - Each feature has its own coefficient

2. **Feature Engineering:**
   - Creating new features from existing ones
   - Encoding categorical variables
   - Selecting relevant features

3. **Model Evaluation:**
   - Use multiple metrics (MAE, RMSE, R², MAPE)
   - Check for overfitting
   - Analyze residuals
   - Interpret feature importance

4. **Best Practices:**
   - Always explore data first (EDA)
   - Handle categorical variables properly
   - Split data before any processing
   - Evaluate on unseen test data
   - Interpret results in business context

---

##  Next Steps

1. **Feature Selection:**
   - Try Lasso regression for automatic feature selection
   - Use correlation analysis to remove redundant features

2. **Model Improvement:**
   - Try Ridge regression for better generalization
   - Experiment with polynomial features
   - Use cross-validation for robust evaluation

3. **Advanced Techniques:**
   - One-Hot Encoding for categorical variables
   - Feature scaling/normalization
   - Interaction terms between features

---

**Congratulations! **  
You've successfully built a multiple linear regression model!